In [1]:
# 02_panel_cleaning.ipynb

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

RAW_PATH = Path("../../data/raw/final_crop_dataset.csv")   # change if still xlsx
PROCESSED_DIR = Path("../../data/processed/CY-Bench-01")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Loading raw data

if RAW_PATH.suffix == ".csv":
    df = pd.read_csv(RAW_PATH)
elif RAW_PATH.suffix in [".xlsx", ".xls"]:
    df = pd.read_excel(RAW_PATH)
else:
    raise ValueError(f"Unsupported file type: {RAW_PATH.suffix}")

print(df.shape)
df.head()

(6715, 28)


,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,awc,bulk_density,drainage_class,latitude,longitude,region_area,crop_area,crop_area_percentage,avg_tmin,avg_tmax,avg_tavg,avg_rad,avg_et0,avg_vpd,avg_cwb,avg_ssm,avg_rsm,avg_ndvi,avg_fpar
0,wheat,IN,IN-14-0001,2003,0.770,15050,19550,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,16.711417,30.860150,23.710323,1.913480e+07,4.349425,32.732055,-3.986236,3.784343,254.632300,0.330688,27.080077
1,wheat,IN,IN-14-0001,2004,0.605,11590,19160,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,15.942244,29.781677,22.634142,1.934158e+07,4.233874,30.660197,-3.651843,5.570850,292.739850,0.408625,35.197833
2,wheat,IN,IN-14-0001,2005,0.605,11590,19160,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,16.505570,30.305594,23.347766,1.931177e+07,4.289164,31.543805,-3.535000,4.622133,265.181766,0.378812,31.107846
3,wheat,IN,IN-14-0001,2006,0.754,14400,19090,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,15.556709,30.182913,22.754858,1.971577e+07,4.289583,31.487173,-3.834213,4.938567,269.223709,0.373188,31.081538
4,wheat,IN,IN-14-0001,2007,0.963,19430,20170,339.086,101.91,10.71,1.591,4,21.45938,81.25028,2296,93.6,4.076,16.346071,30.354496,23.244661,1.954419e+07,4.294756,31.479134,-4.076094,5.065094,272.097992,0.351313,29.904000


In [3]:
# Standardizing the column names

rename_map = {
    "cropname": "crop_name",
    "countrycode": "country_code",
    "admid": "adm_id",
    "harvestyear": "harvest_year",
    "harvestarea": "harvest_area",
    "drainageclass": "drainage_class",
    "bulkdensity": "bulk_density",
    "regionarea": "region_area",
    "croparea": "crop_area",
    "cropareapercentage": "crop_area_percentage",
    "avgtmin": "avg_tmin",
    "avgtmax": "avg_tmax",
    "avgtavg": "avg_tavg",
    "avgrad": "avg_rad",
    "avget0": "avg_et0",
    "avgvpd": "avg_vpd",
    "avgcwb": "avg_cwb",
    "avgssm": "avg_ssm",
    "avgrsm": "avg_rsm",
    "avgndvi": "avg_ndvi",
    "avgfpar": "avg_fpar",
}

df = df.rename(columns=rename_map)
df.columns = [c.strip().lower() for c in df.columns]

sorted(df.columns)

['adm_id',
 'avg_cwb',
 'avg_et0',
 'avg_fpar',
 'avg_ndvi',
 'avg_rad',
 'avg_rsm',
 'avg_ssm',
 'avg_tavg',
 'avg_tmax',
 'avg_tmin',
 'avg_vpd',
 'awc',
 'bulk_density',
 'country_code',
 'crop_area',
 'crop_area_percentage',
 'crop_name',
 'drainage_class',
 'eos',
 'harvest_area',
 'harvest_year',
 'latitude',
 'longitude',
 'production',
 'region_area',
 'sos',
 'yield']

In [4]:
# Expected columns check

expected_cols = [
    "crop_name", "country_code", "adm_id", "harvest_year", "yield",
    "production", "harvest_area", "sos", "eos",
    "awc", "bulk_density", "drainage_class", "latitude", "longitude",
    "region_area", "crop_area", "crop_area_percentage",
    "avg_tmin", "avg_tmax", "avg_tavg", "avg_rad", "avg_et0", "avg_vpd",
    "avg_cwb", "avg_ssm", "avg_rsm", "avg_ndvi", "avg_fpar"
]

missing = [c for c in expected_cols if c not in df.columns]
extra = [c for c in df.columns if c not in expected_cols]

print("Missing:", missing)
print("Extra:", extra)

Missing: []
Extra: []


In [5]:
# Enforcing types

id_cols = ["crop_name", "country_code", "adm_id"]
for c in id_cols:
    df[c] = df[c].astype(str).str.strip()

df["harvest_year"] = pd.to_numeric(df["harvest_year"], errors="coerce").astype("Int64")

numeric_cols = [c for c in df.columns if c not in id_cols + ["harvest_year"]]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6715 entries, 0 to 6714
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   crop_name             6715 non-null   str    
 1   country_code          6715 non-null   str    
 2   adm_id                6715 non-null   str    
 3   harvest_year          6715 non-null   Int64  
 4   yield                 6715 non-null   float64
 5   production            6715 non-null   int64  
 6   harvest_area          6715 non-null   int64  
 7   sos                   6715 non-null   float64
 8   eos                   6715 non-null   float64
 9   awc                   6715 non-null   float64
 10  bulk_density          6715 non-null   float64
 11  drainage_class        6715 non-null   int64  
 12  latitude              6715 non-null   float64
 13  longitude             6715 non-null   float64
 14  region_area           6715 non-null   int64  
 15  crop_area             6715 non-n

In [6]:
# Basic row hygiene

df = df.drop_duplicates()
df = df.sort_values(["crop_name", "adm_id", "harvest_year"]).reset_index(drop=True)

print(df.shape)
df.head()

(6715, 28)


,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,awc,bulk_density,drainage_class,latitude,longitude,region_area,crop_area,crop_area_percentage,avg_tmin,avg_tmax,avg_tavg,avg_rad,avg_et0,avg_vpd,avg_cwb,avg_ssm,avg_rsm,avg_ndvi,avg_fpar
0,wheat,IN,IN-01-0047,2017,1.000,10,10,320.213,92.669,14.203,1.506,4,17.52128,81.18046,8316,20.2,0.243,20.390616,32.775493,26.082348,1.946257e+07,4.527138,31.032674,-4.371399,3.926333,244.065007,0.569833,54.283143
1,wheat,IN,IN-01-0049,2010,1.500,60,40,284.497,87.670,15.192,1.542,4,16.39034,79.72043,11398,72.4,0.635,21.423054,32.617583,26.163827,1.834995e+07,4.514167,28.304595,-3.802982,4.619137,254.904083,0.577476,55.225625
2,wheat,IN,IN-01-0051,2003,0.304,210,690,313.989,102.880,15.396,1.594,4,15.56859,77.82124,17419,186.8,1.072,21.064903,33.499558,27.157851,2.022468e+07,5.366610,36.000909,-5.287260,2.870662,273.150972,0.369895,33.128000
3,wheat,IN,IN-01-0051,2004,0.231,90,390,313.989,102.880,15.396,1.594,4,15.56859,77.82124,17419,186.8,1.072,20.575006,33.182636,26.735961,2.047566e+07,5.346344,35.987214,-5.182110,2.815935,273.161974,0.367158,33.340625
4,wheat,IN,IN-01-0051,2005,0.592,420,710,313.989,102.880,15.396,1.594,4,15.56859,77.82124,17419,186.8,1.072,20.696252,33.293755,26.864381,2.056602e+07,5.275374,35.228348,-5.073142,2.813116,273.477381,0.387053,35.966375


In [7]:
# Missingness overview

missing_summary = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .rename("missing_fraction")
      .reset_index()
      .rename(columns={"index": "column"})
)

missing_summary.head(20)

,column,missing_fraction
0,crop_name,0.0
1,country_code,0.0
2,adm_id,0.0
3,harvest_year,0.0
4,yield,0.0
5,production,0.0
6,harvest_area,0.0
7,sos,0.0
8,eos,0.0
9,awc,0.0


In [8]:
# Checking uniqueness of panel key

key_dupes = (
    df.groupby(["adm_id", "harvest_year"])
      .size()
      .reset_index(name="n")
      .query("n > 1")
)

print("Duplicate adm_id-year rows:", len(key_dupes))
key_dupes.head()

Duplicate adm_id-year rows: 0


,adm_id,harvest_year,n


In [9]:
# Panel length summary by unit

unit_summary = (
    df.groupby("adm_id")
      .agg(
          crop_name=("crop_name", "first"),
          first_year=("harvest_year", "min"),
          last_year=("harvest_year", "max"),
          n_years=("harvest_year", "nunique"),
          n_rows=("harvest_year", "size"),
      )
      .reset_index()
)

unit_summary["is_balanced_15"] = unit_summary["n_years"].eq(15)
unit_summary.describe(include="all")

,adm_id,crop_name,first_year,last_year,n_years,n_rows,is_balanced_15
count,501,501,501.0,501.0,501.000000,501.0,501
unique,501,1,<NA>,<NA>,NaN,<NA>,2
top,IN-01-0047,wheat,<NA>,<NA>,NaN,<NA>,True
freq,1,501,<NA>,<NA>,NaN,<NA>,385
mean,NaN,NaN,2004.171657,2016.670659,13.403194,13.403194,NaN
std,NaN,NaN,3.185981,1.430146,3.775329,3.775329,NaN
min,NaN,NaN,2003.0,2008.0,1.000000,1.0,NaN
25%,NaN,NaN,2003.0,2017.0,15.000000,15.0,NaN
50%,NaN,NaN,2003.0,2017.0,15.000000,15.0,NaN
75%,NaN,NaN,2003.0,2017.0,15.000000,15.0,NaN


In [10]:
# Static vs dynamic variable checks

static_candidates = [
    "country_code", "latitude", "longitude", "awc", "bulk_density",
    "drainage_class", "region_area", "crop_area", "crop_area_percentage",
    "sos", "eos"
]

static_check = []
for col in static_candidates:
    nunique_within_unit = df.groupby("adm_id")[col].nunique(dropna=False)
    static_check.append({
        "column": col,
        "max_unique_within_adm": nunique_within_unit.max(),
        "n_units_varying": int((nunique_within_unit > 1).sum())
    })

static_check = pd.DataFrame(static_check).sort_values(
    ["n_units_varying", "max_unique_within_adm"], ascending=[False, False]
)

static_check

,column,max_unique_within_adm,n_units_varying
0,country_code,1,0
1,latitude,1,0
2,longitude,1,0
3,awc,1,0
4,bulk_density,1,0
5,drainage_class,1,0
6,region_area,1,0
7,crop_area,1,0
8,crop_area_percentage,1,0
9,sos,1,0


In [11]:
# keep only one crop if dataset is single-crop or if you want one crop panel
# this cell is optional, not needed if there is a single crop data in the dataset

df["crop_name"].value_counts(dropna=False).head(20)

crop_name
wheat    6715
Name: count, dtype: int64

In [12]:
# Domain sanity checks

domain_rules = {
    "yield": (0, None),
    "production": (0, None),
    "harvest_area": (0, None),
    "avg_ndvi": (0, 1),
    "avg_ssm": (0, None),
    "avg_fpar": (0, 100),
}

sanity_rows = []
for col, (low, high) in domain_rules.items():
    s = df[col].dropna()
    bad_low = int((s < low).sum()) if low is not None else 0
    bad_high = int((s > high).sum()) if high is not None else 0
    sanity_rows.append({
        "column": col,
        "below_min": bad_low,
        "above_max": bad_high,
        "min_observed": s.min() if len(s) else np.nan,
        "max_observed": s.max() if len(s) else np.nan,
    })

sanity_report = pd.DataFrame(sanity_rows)
sanity_report

,column,below_min,above_max,min_observed,max_observed
0,yield,0,0,0.107000,5.860000e+00
1,production,0,0,0.000000,1.969000e+06
2,harvest_area,0,0,0.000000,3.970000e+05
3,avg_ndvi,0,0,0.092222,8.289474e-01
4,avg_ssm,0,0,0.793918,8.382297e+00
5,avg_fpar,0,0,3.880500,9.004000e+01


In [13]:
# Building ordered variable groups for downstream use

id_vars = ["crop_name", "country_code", "adm_id", "harvest_year"]
# for now only one country is there

static_vars = [
    "latitude", "longitude", "awc", "bulk_density", "drainage_class",
    "region_area", "crop_area", "crop_area_percentage", "sos", "eos"
]

outcome_vars = ["yield", "production", "harvest_area"]

climate_vars = [
    "avg_tmin", "avg_tmax", "avg_tavg", "avg_rad", "avg_et0", "avg_vpd", "avg_cwb"
]

water_vars = ["avg_ssm", "avg_rsm"]

vegetation_vars = ["avg_ndvi", "avg_fpar"]

ordered_cols = id_vars + static_vars + outcome_vars + climate_vars + water_vars + vegetation_vars
ordered_cols = [c for c in ordered_cols if c in df.columns]

clean_panel = df[ordered_cols].copy()
clean_panel.head()

,crop_name,country_code,adm_id,harvest_year,latitude,longitude,awc,bulk_density,drainage_class,region_area,crop_area,crop_area_percentage,sos,eos,yield,production,harvest_area,avg_tmin,avg_tmax,avg_tavg,avg_rad,avg_et0,avg_vpd,avg_cwb,avg_ssm,avg_rsm,avg_ndvi,avg_fpar
0,wheat,IN,IN-01-0047,2017,17.52128,81.18046,14.203,1.506,4,8316,20.2,0.243,320.213,92.669,1.000,10,10,20.390616,32.775493,26.082348,1.946257e+07,4.527138,31.032674,-4.371399,3.926333,244.065007,0.569833,54.283143
1,wheat,IN,IN-01-0049,2010,16.39034,79.72043,15.192,1.542,4,11398,72.4,0.635,284.497,87.670,1.500,60,40,21.423054,32.617583,26.163827,1.834995e+07,4.514167,28.304595,-3.802982,4.619137,254.904083,0.577476,55.225625
2,wheat,IN,IN-01-0051,2003,15.56859,77.82124,15.396,1.594,4,17419,186.8,1.072,313.989,102.880,0.304,210,690,21.064903,33.499558,27.157851,2.022468e+07,5.366610,36.000909,-5.287260,2.870662,273.150972,0.369895,33.128000
3,wheat,IN,IN-01-0051,2004,15.56859,77.82124,15.396,1.594,4,17419,186.8,1.072,313.989,102.880,0.231,90,390,20.575006,33.182636,26.735961,2.047566e+07,5.346344,35.987214,-5.182110,2.815935,273.161974,0.367158,33.340625
4,wheat,IN,IN-01-0051,2005,15.56859,77.82124,15.396,1.594,4,17419,186.8,1.072,313.989,102.880,0.592,420,710,20.696252,33.293755,26.864381,2.056602e+07,5.275374,35.228348,-5.073142,2.813116,273.477381,0.387053,35.966375


In [14]:
# Saving processed outputs

clean_panel.to_csv(PROCESSED_DIR / "clean_panel.csv", index=False)
unit_summary.to_csv(PROCESSED_DIR / "unit_summary.csv", index=False)
missing_summary.to_csv(PROCESSED_DIR / "missing_summary.csv", index=False)
static_check.to_csv(PROCESSED_DIR / "static_check.csv", index=False)
sanity_report.to_csv(PROCESSED_DIR / "sanity_report.csv", index=False)

print("Saved:")
for p in [
    PROCESSED_DIR / "clean_panel.csv",
    PROCESSED_DIR / "unit_summary.csv",
    PROCESSED_DIR / "missing_summary.csv",
    PROCESSED_DIR / "static_check.csv",
    PROCESSED_DIR / "sanity_report.csv",
]:
    print("-", p)

Saved:
- ../../data/processed/CY-Bench-01/clean_panel.csv
- ../../data/processed/CY-Bench-01/unit_summary.csv
- ../../data/processed/CY-Bench-01/missing_summary.csv
- ../../data/processed/CY-Bench-01/static_check.csv
- ../../data/processed/CY-Bench-01/sanity_report.csv


In [15]:
# Quick final audit

print("Rows, columns:", clean_panel.shape)
print("Unique units:", clean_panel["adm_id"].nunique())
print("Year range:", clean_panel["harvest_year"].min(), clean_panel["harvest_year"].max())
print("Crops:", clean_panel["crop_name"].unique()[:10])

clean_panel.sample(5, random_state=42)

Rows, columns: (6715, 28)
Unique units: 501
Year range: 2003 2017
Crops: <StringArray>
['wheat']
Length: 1, dtype: str


,crop_name,country_code,adm_id,harvest_year,latitude,longitude,awc,bulk_density,drainage_class,region_area,crop_area,crop_area_percentage,sos,eos,yield,production,harvest_area,avg_tmin,avg_tmax,avg_tavg,avg_rad,avg_et0,avg_vpd,avg_cwb,avg_ssm,avg_rsm,avg_ndvi,avg_fpar
1193,wheat,IN,IN-04-0188,2014,28.36921,77.32745,10.980,1.557,5,744,316.9,42.598,342.715,96.385,3.321,93000,28000,11.275866,22.626580,16.605697,1.587013e+07,3.051756,15.983706,-1.900521,3.719798,185.341899,0.539333,54.985000
3864,wheat,IN,IN-10-0164,2009,24.46010,73.83956,12.844,1.531,5,11721,953.2,8.132,341.685,112.776,2.341,75370,32200,16.397504,30.171401,23.228117,1.965838e+07,4.934445,34.555496,-4.907642,3.206299,214.733460,0.302647,28.297571
681,wheat,IN,IN-03-0125,2013,22.27926,73.21068,14.392,1.538,5,4093,132.4,3.234,338.517,100.840,2.432,71620,29450,17.666109,32.075758,24.498164,1.989798e+07,5.019867,37.305453,-5.016125,3.039922,178.122328,0.477250,41.826417
3707,wheat,IN,IN-10-0153,2017,25.15459,72.24645,12.859,1.448,6,10656,815.6,7.654,341.991,111.903,2.145,100700,46940,15.937081,30.449728,23.028331,1.977260e+07,5.006735,34.803485,-4.988051,2.238934,195.578890,0.329059,33.224214
3575,wheat,IN,IN-10-0145,2005,28.21983,73.40939,9.730,1.441,6,30253,3867.8,12.785,340.240,120.469,1.864,93500,50160,13.895877,26.680062,20.474938,1.807025e+07,4.686027,27.811377,-4.323541,1.750986,103.435110,0.141500,13.161143
